<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/6_Neo4j_Contexto_Relacional_AprenderHaciendo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir S6-ALT en Colab"></a>

**Acceso público:** [página del curso](https://jazaineam1.github.io/BigData2026/) · **Laboratorio guiado (checklist paso a paso):** [ábrelo en otra pestaña ↗](https://jazaineam1.github.io/BigData2026/assets/tutoriales/s06-laboratorio-guiado.html)

> **Versión alterna de S6**, con el mismo contenido de fondo que `6_Neo4j_Contexto_Relacional.ipynb` pero en ciclos más cortos de concepto → acción, y trabajando siempre sobre un **caso compartido** (no requiere el archivo de S5). Es una comparación pedagógica, no la versión oficial.

In [ ]:
#@title Preparar interactividad { display-mode: "form" }
import base64, json, html as html_lib
from IPython.display import display, HTML

def pregunta_codificada(token):
    p = json.loads(base64.b64decode(token).decode("utf-8"))
    uid = f"s06-p{p['numero']}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>'
        for i, op in enumerate(p["opciones"])
    )
    # Escapar el atributo COMPLETO; la retroalimentación se inserta como texto.
    handler = (
        "const box=this.closest('[data-pregunta]');"
        "const e=box.querySelector('input:checked');"
        "const s=box.querySelector('[aria-live]');"
        "if(!e){s.textContent='Selecciona una opción.';return;}"
        f"const i=Number(e.value),r={json.dumps(p['retro'], ensure_ascii=False)};"
        f"const ok=i==={p['correcta']};"
        "s.textContent=(ok?'Correcto. ':'Incorrecto. ')+r[i];"
        "s.style.background=ok?'#dcfce7':'#fee2e2';"
        "s.style.color=ok?'#14532d':'#7f1d1d';"
        "s.style.padding='12px';"
    )
    handler = html_lib.escape(handler, quote=True)
    contador = str(p['numero']) + (f" de {p['total']}" if p.get('total') else '')
    box = (
        f'<div data-pregunta="{uid}" style="border:2px solid #1e40af;background:#eff6ff;color:#172554;border-radius:12px;padding:15px;margin:14px 0">'
        f'<strong>Pregunta {contador} — {html_lib.escape(p["tema"])}</strong>'
        f'<p style="background:#fef3c7;color:#713f12;padding:10px">{html_lib.escape(p.get("contexto", "Aplica lo que acabas de observar en el caso de Laura."))}</p>'
        f'<p>{html_lib.escape(p["pregunta"])}</p>{opts}'
        f'<button onclick="{handler}" '
        f'style="background:#1e40af;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar respuesta</button>'
        f'<div id="r-{uid}" aria-live="polite"></div></div>'
    )
    display(HTML(box))

def tutorial(url, alto=720):
    box = f'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>'
    box += f'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>'
    display(HTML(box))

print("Soporte S6 listo.")

# Sesión 6 (alterna) — De la fila priorizada al contexto relacional con Neo4j

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos — BIG DATA (64491093)

**Caso conductor:** Compras Claras
**Pregunta profesional:** **Laura ya sabe qué proceso revisar primero. Antes de asignarlo a un auditor, ¿qué relaciones alrededor de ese proceso necesita ver para comprender su contexto?**

**Respuesta corta y herramienta:** Laura necesita seguir una cadena de conexiones —entidad → proceso → proveedor → otro proceso → otra entidad— sin perder el hilo en ningún salto. Eso es una pregunta sobre relaciones que se conectan entre sí, y la herramienta que la responde es una **base de datos de grafos: Neo4j**.

### Producto observable

Al terminar tendrás una **ficha relacional de revisión** con:

1. el proceso de trabajo de esta sesión y su contexto de prensa;
2. procesos históricos adjudicados de su entidad;
3. proveedores y otras entidades conectadas cuando el dato lo sostenga;
4. el contraste de H2-R, la hipótesis relacional de hoy (pandas ↔ Neo4j);
5. una decisión de modelado y una alternativa descartada;
6. un límite concreto;
7. `s06_contexto_procesos.jsonl`, entrada de la siguiente sesión.

**OJO — diferencia con la versión original.** Esta versión trabaja siempre sobre la misma entidad (el caso más rico del extracto), no sobre el proceso que elegiste en S5. Ganas comparabilidad entre compañeros; pierdes la individualización total. Tu decisión sobre qué proveedor explorar, tu límite y tu alternativa siguen siendo tuyos.

## El hilo del evaluador



**Cómo se lee.** Cada sesión entrega el producto que abre la siguiente.

**Qué nos dice.** S6 continúa la misma investigación, no empieza un tema suelto.

**Qué NO permite concluir todavía.** Que exista una cadena de sesiones no significa que ya haya evidencia de algo irregular.

**Error frecuente.** Tratar cada sesión como un capítulo aislado.

## Neo4j: qué es, y por qué aparece aquí

### Qué es, en una frase

**Neo4j guarda la relación misma como un dato** —no como un cálculo que se rehace cada vez que preguntas—, para poder recorrer varios saltos de conexión sin escribir un cruce por cada salto.

### Cómo lo logra

Cada nodo (`Entidad`, `Proceso`, `Proveedor`) y cada relación (`PUBLICA`, `ADJUDICADO_A`) quedan guardados juntos. Preguntar “¿qué hay conectado a esto, y qué hay conectado a eso otro?” es recorrer flechas ya guardadas, no repetir un cruce por cada nivel de la cadena.

### Frente a lo que ya conocías

| Motor | Cómo resuelve “cruzar” | Costo de un salto adicional |
|---|---|---|
| MongoDB (`$lookup`) / SQL (`JOIN`) | recalcula el cruce en cada consulta | crece con cada nivel que agregas |
| Cassandra (S5) | evita el cruce: diseña la tabla para una sola pregunta fija | no aplica — esa pregunta no cambia |
| Neo4j (hoy) | guarda la relación como dato y la recorre | un salto más es una flecha más, no un cruce más |

### ¿Dónde se usa esto en la vida real?

| Sistema que ya conoces | Nodo | Relación | Pregunta que responde |
|---|---|---|---|
| LinkedIn / Facebook | una persona | `ES_AMIGO_DE`, `TRABAJA_EN` | “¿Cuál es el contacto en común entre tú y un desconocido?” |
| Google Maps / Waze | una intersección | `CONECTA_CON` | “¿Cuál es la ruta más corta entre dos puntos?” |
| Netflix / Spotify | un usuario o contenido | `VIO`, `ESCUCHÓ` | “¿Qué otros usuarios con gustos parecidos vieron algo que tú no?” |
| Un banco antifraude | una cuenta | `TRANSFIRIÓ_A` | “¿Esta cuenta está conectada, en pocos saltos, con cuentas ya marcadas?” |

Hoy tu grafo es más pequeño (`Entidad`, `Proceso`, `Proveedor`), pero la pregunta es la misma familia: **quién está conectado con quién, y a través de qué**.

In [ ]:
#@title Autoevaluación 1 — Motor { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAxLCAidGVtYSI6ICJNb3RvciIsICJwcmVndW50YSI6ICLCv0N1w6FsIGRlIGVzdG9zIG1vdG9yZXMgcmVjYWxjdWxhIGVsIGNydWNlIGNvbXBsZXRvIGNhZGEgdmV6IHF1ZSBwcmVndW50YXMsIGVuIHZleiBkZSBndWFyZGFyIGxhIHJlbGFjacOzbiBjb21vIHBhcnRlIGRlbCBkYXRvPyIsICJvcGNpb25lcyI6IFsiTmVvNGouIiwgIk1vbmdvREIgKGAkbG9va3VwYCkgbyBTUUwgKGBKT0lOYCkuIiwgIkNhc3NhbmRyYS4iXSwgImNvcnJlY3RhIjogMSwgInJldHJvIjogWyJOZW80aiBndWFyZGEgbGEgcmVsYWNpw7NuIGNvbW8gZGF0bzsgcG9yIGVzbyBubyBsYSByZWNhbGN1bGEuIiwgIkNvcnJlY3RvLiBDYWRhIGNvbnN1bHRhIGNvbiAkbG9va3VwIG8gSk9JTiB2dWVsdmUgYSBjcnV6YXIgZGVzZGUgY2Vyby4iLCAiQ2Fzc2FuZHJhIGV2aXRhIGVsIGNydWNlIGRpc2XDsWFuZG8gbGEgdGFibGEgcGFyYSB1bmEgc29sYSBwcmVndW50YSBmaWphLiJdLCAiY29udGV4dG8iOiAiQXBsaWNhIGxvIHF1ZSBhY2FiYXMgZGUgb2JzZXJ2YXIgZW4gZWwgY2FzbyBkZSBMYXVyYS4iLCAidG90YWwiOiBudWxsfQ==")

**PARA LLEVAR.** Neo4j no aparece porque “toca grafos”. Aparece porque la pregunta de Laura —qué hay alrededor de este proceso, y alrededor de eso— ya es, literalmente, una pregunta de relaciones.

## Mapa de la sesión

| Bloque | Rol | Pregunta | Herramienta | Qué queda |
|---|---|---|---|---|
| 1. El caso de trabajo | 🧠 ENTIENDE | ¿qué proceso y qué historial vamos a usar? | Colab | proceso + entidad + prensa |
| 2. H2-R y contexto | 🧠 + ✏️ | ¿qué historial rodea esa entidad? | pandas | tabla de contraste |
| 3. Diseñar | 🧠 + ✏️ | ¿qué es nodo y qué es relación? | papel + cuaderno | Entidad → Proceso → Proveedor |
| 4. Contrato pandas | ▶️ EJECUTA | ¿qué debe responder el grafo? | pandas | resultado esperado |
| 5. AuraDB | ▶️ EJECUTA | ¿cómo levantamos el servicio? | tutorial + Neo4j Aura | conexión real |
| 6. Cypher | ▶️ EJECUTA | ¿cómo cargamos y recorremos relaciones? | Cypher/Neo4j | grafo consultable |
| 7. Verificar | 🧠 + ▶️ | ¿Neo4j conserva la respuesta? | pandas + Neo4j | pandas = Neo4j |
| 8. Hito | ✏️ MODIFICA | ¿qué puede sostener Laura? | Colab | ficha + límite + export |

### Semáforo de código

- 🧠 **ENTIENDE:** debes poder explicarlo con tus palabras.
- ▶️ **EJECUTA:** corre la celda y verifica la salida; **no necesitas escribirla de memoria**.
- ✏️ **MODIFICA/DECIDE:** cambia el dato señalado o responde antes de ver el resultado completo.

**Diferencia con la versión original de S6:** aquí casi todos los bloques mezclan 🧠 y ✏️ en la misma frase — nunca hay dos conceptos seguidos sin una acción entre medio.

---
## 1. El caso de trabajo de esta sesión

Esta versión trabaja siempre sobre la misma entidad: la más rica en relaciones de todo el extracto. No pide ningún archivo de S5 — así cualquiera puede correr este cuaderno de forma independiente.

**OJO.** El hito declara con honestidad que este es el caso compartido, no un proceso que tú elegiste en S5.

In [ ]:
import json
import urllib.request
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'
MANIFEST_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json'

datos = pd.read_csv(DATA_URL, low_memory=False)
with urllib.request.urlopen(MANIFEST_URL) as r:
    manifest = json.loads(r.read().decode("utf-8"))

ancla_trabajo = dict(manifest["ancla_pedagogica"])
print("Entidad de trabajo:", ancla_trabajo["entidad"])
print(json.dumps(ancla_trabajo, ensure_ascii=False, indent=2))
print("Filas disponibles:", len(datos))

### Cómo se lee la entrada

**Cómo se lee.** El caso de trabajo es un proceso real de SECOP, elegido porque su entidad tiene el historial más rico del extracto — así el ejercicio siempre tiene señal suficiente para explorar.

**Qué nos dice.** Trabajamos sobre datos reales desde la primera celda.

**Qué NO permite concluir todavía.** Tener historial contractual no significa que exista una relación problemática.

**Error frecuente.** Pensar que esta entidad fue elegida por sospecha — se eligió por tener suficiente historial para aprender, nada más.

---
## 2. H2-R: la hipótesis relacional de esta sesión

S5 cerró con una hipótesis de prensa: **H1** — ¿aparece literalmente alguno de los 77 IDs de proceso en título o subtítulo de una noticia? El resultado fue `0/77`: **H1 literal refutada**, y la prensa quedó especificada como contexto de entidad, no como evidencia directa de un proceso.

S6 abre una hipótesis distinta, ahora relacional:

> **H2-R.** El proveedor histórico más conectado de la entidad de este caso está conectado con más entidades que la mediana de esa misma conexión entre las 32 entidades candidatas de S5 que tienen historial.

No es una prueba estadística inferencial: es una comparación empírica y falsable sobre este extracto.

### Ejemplo manual pequeño (nombres inventados, para pensar antes de programar)



Mirando solo este dibujo, sin ninguna tabla: **Constructora Ejemplo S.A.S. aparece conectada con dos entidades distintas** —la Alcaldía de Ejemplo y la Gobernación de Prueba— a través de dos procesos separados. Eso es H2-R con nombres inventados: Constructora Ejemplo S.A.S. es justo el tipo de proveedor que la hipótesis busca — uno adjudicado por más de una entidad.

El proceso de este caso puede no estar adjudicado directamente: **no le inventamos un proveedor**. El historial adjudicado —procesos ya cerrados de la misma entidad— es lo que sí aporta proveedores reales y conexiones observadas.

In [ ]:
#@title Autoevaluación 2 — H2-R { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAyLCAidGVtYSI6ICJIMi1SIiwgInByZWd1bnRhIjogIlNpIENvbnN0cnVjdG9yYSBFamVtcGxvIFMuQS5TLiBhcGFyZWNlIGFkanVkaWNhZGEgdGFudG8gcG9yIGxhIEFsY2FsZMOtYSBkZSBFamVtcGxvIGNvbW8gcG9yIGxhIEdvYmVybmFjacOzbiBkZSBQcnVlYmEsIMK/cXXDqSBwdWVkZSBhZmlybWFyc2U/IiwgIm9wY2lvbmVzIjogWyJRdWUgQ29uc3RydWN0b3JhIEVqZW1wbG8gUy5BLlMuIGFwYXJlY2UgY29uZWN0YWRhIGNvbnRyYWN0dWFsbWVudGUgY29uIGFsIG1lbm9zIGRvcyBlbnRpZGFkZXMgZW4gZXN0ZSBleHRyYWN0by4iLCAiUXVlIENvbnN0cnVjdG9yYSBFamVtcGxvIFMuQS5TLiBlcyBzb3NwZWNob3NhIGRlIGlycmVndWxhcmlkYWQuIiwgIlF1ZSBsYXMgZG9zIGVudGlkYWRlcyBjb29yZGluYXJvbiBsYSBhZGp1ZGljYWNpw7NuIGVudHJlIHPDrS4iXSwgImNvcnJlY3RhIjogMCwgInJldHJvIjogWyJDb3JyZWN0by4gRXNvIGVzIGV4YWN0YW1lbnRlIGxvIHF1ZSBlbCBncmFmbyBkZXNjcmliZTogdW5hIGVzdHJ1Y3R1cmEgcmVnaXN0cmFkYSwgbm8gdW5hIGNvbmNsdXNpw7NuIHNvYnJlIGNvbmR1Y3RhLiIsICJMYSBjb25lY3RpdmlkYWQgcG9yIHPDrSBzb2xhIG5vIHBydWViYSBpcnJlZ3VsYXJpZGFkLiIsICJMYSBjb25lY3RpdmlkYWQgcG9yIHPDrSBzb2xhIG5vIHBydWViYSBjb29yZGluYWNpw7NuLiJdLCAiY29udGV4dG8iOiAiQXBsaWNhIGxvIHF1ZSBhY2FiYXMgZGUgb2JzZXJ2YXIgZW4gZWwgY2FzbyBkZSBMYXVyYS4iLCAidG90YWwiOiBudWxsfQ==")

In [ ]:
nit_deseado = str(ancla_trabajo.get("nit_entidad", "")).strip()
hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
hist_ancla = hist[hist["nit_entidad"].astype(str).str.strip().eq(nit_deseado)]

print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Procesos históricos:", hist_ancla["id_proceso"].nunique())
print("Proveedores distintos:", hist_ancla["nit_proveedor"].nunique())

### Interpretación del contexto histórico

**Cómo se lee.** Los conteos corresponden al historial adjudicado disponible para la entidad de trabajo, no al proceso candidato aislado.

**Qué nos dice.** Hay material relacional suficiente para preguntar por proveedores y conexiones entre procesos.

**Qué NO permite concluir todavía.** Más procesos o proveedores no equivalen a mayor riesgo.

**Error frecuente.** Usar el número de contratos como una puntuación de sospecha.

In [ ]:
#@title Autoevaluación 3 — Modelo { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAzLCAidGVtYSI6ICJNb2RlbG8iLCAicHJlZ3VudGEiOiAiQWNhYmFzIGRlIHZlciBlbCBoaXN0b3JpYWwgZGUgZXN0YSBlbnRpZGFkLiDCv1BvciBxdcOpIGVsIHByb2Nlc28gZGUgdHJhYmFqbyBubyBuZWNlc2l0YSB0b2RhdsOtYSB1bmEgcmVsYWNpw7NuIGhhY2lhIHVuIHByb3ZlZWRvcj8iLCAib3BjaW9uZXMiOiBbIlBvcnF1ZSBOZW80aiBubyBzb3BvcnRhIHByb3ZlZWRvcmVzIGVuIHByb2Nlc29zIHJlY2llbnRlcy4iLCAiUG9ycXVlIHB1ZWRlIG5vIGVzdGFyIGFkanVkaWNhZG8gZGlyZWN0YW1lbnRlOyBzaXJ2ZSBjb21vIGFuY2xhIHkgZWwgaGlzdG9yaWFsIGFwb3J0YSBwcm92ZWVkb3JlcyByZWFsZXMuIiwgIlBvcnF1ZSBsb3MgcHJvdmVlZG9yZXMgcGVydGVuZWNlbiBhIEVsYXN0aWNzZWFyY2guIl0sICJjb3JyZWN0YSI6IDEsICJyZXRybyI6IFsiTmVvNGogc8OtIHNvcG9ydGEgZXNhIHJlbGFjacOzbjsgZWwgbMOtbWl0ZSBlc3TDoSBlbiBsYSBldmlkZW5jaWEuIiwgIkV4YWN0by4gTm8gZmFicmljYW1vcyB1bmEgcmVsYWNpw7NuIHF1ZSBlbCBkYXRvIG5vIHNvc3RpZW5lLiIsICJFbGFzdGljc2VhcmNoIHJlc29sdmVyw6Egb3RyYSBwcmVndW50YTogYsO6c3F1ZWRhIHRleHR1YWwgeSByZWxldmFuY2lhLiJdLCAiY29udGV4dG8iOiAiQXBsaWNhIGxvIHF1ZSBhY2FiYXMgZGUgb2JzZXJ2YXIgZW4gZWwgY2FzbyBkZSBMYXVyYS4iLCAidG90YWwiOiBudWxsfQ==")

---
## 3. Diseñar el grafo antes de escribir la consulta final

Antes de escribir Cypher, cinco palabras y nada más.

| Concepto | Qué es | Ejemplo (Constructora Ejemplo S.A.S.) |
|---|---|---|
| Nodo | una entidad del dominio | el proveedor mismo |
| Label | la categoría del nodo | `Proveedor` |
| Propiedad | un dato guardado en el nodo | `nit: "900123456"` |
| Relación | un hecho dirigido entre dos nodos | `ADJUDICADO_A` |
| Camino | una secuencia de nodos y relaciones | Entidad → Proceso → Proveedor |

**Los mismos 5 conceptos, en LinkedIn:** Nodo = una persona · Label = `Persona` o `Empresa` · Propiedad = `nombre: "Ana"` · Relación = `ES_CONTACTO_DE` · Camino = la cadena de contactos que te conecta con alguien que nunca has visto.

In [ ]:
# EJERCICIO — identifica las tres piezas del patrón (v:Proveedor {nit:"900123456"})
variable = "____"   # una sola letra: la variable con la que nombras el nodo
label = "____"      # la categoria (el label) del nodo
propiedad = "____"  # el par clave:valor, tal cual aparece entre llaves

if variable != "v":
    raise ValueError('La variable es la letra justo despues del parentesis: v')
if label != "Proveedor":
    raise ValueError('El label es la categoria despues de los dos puntos: Proveedor')
if propiedad != 'nit:"900123456"':
    raise ValueError('La propiedad es el par completo entre llaves: nit:"900123456"')
print("Correcto: variable=v, label=Proveedor, propiedad=nit:\"900123456\"")

### Cómo se lee `(e:Entidad {nit:"123"})`

- `e` — variable con la que nombras este nodo en el resto de la consulta.
- `Entidad` — el label: la categoría a la que pertenece.
- `{nit:"123"}` — una propiedad que identifica cuál Entidad exactamente.

Con eso ya puedes leer un patrón completo: `(e:Entidad)-[:PUBLICA]->(p:Proceso)` es "un nodo Entidad conectado, mediante la relación PUBLICA, a un nodo Proceso".

### EJERCICIO S06-PATRON — identifica la relación

Lee el nombre de la relación entre un proceso histórico y el proveedor al que fue adjudicado.

**Qué debe verse:** `(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)`.
**Error común:** confundir el ID de un proceso con el nombre de la relación. El ID identifica el nodo; `ADJUDICADO_A` nombra la flecha.

<details><summary><strong>Recuperación si te atascaste</strong></summary>
La relación se llama <code>ADJUDICADO_A</code>; el cuaderno ya la deja preparada.
</details>

In [ ]:
RELACION_PROCESO_PROVEEDOR = "ADJUDICADO_A"
patron_estudiante = f"(p:Proceso)-[:{RELACION_PROCESO_PROVEEDOR}]->(v:Proveedor)"
print(patron_estudiante)

print("Patrón correcto: ADJUDICADO_A expresa una adjudicación observada.")

### Modelo mínimo que usaremos



`(e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)` — así se escribe ese mismo dibujo en Cypher.

| Elemento | Identificador | Decisión |
|---|---|---|
| `Entidad` | NIT | actor que publica |
| `Proceso` | ID SECOP | nodo con texto, valor, modalidad y URL |
| `Proveedor` | NIT | actor adjudicado que puede conectar procesos |
| `PUBLICA` | relación | quién publica el proceso |
| `ADJUDICADO_A` | relación | a quién se adjudicó un proceso histórico |

### La alternativa que descartamos

| Opción | Cómo se vería | Por qué no la usamos hoy |
|---|---|---|
| **Proceso como nodo** (la que usamos) | `(e)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v)` | `Proceso` participa en caminos propios y S7 reutiliza su texto |
| Proceso como propiedad de una relación directa | `(e:Entidad)-[:CONTRATO {id_proceso:"...", valor:...}]->(v:Proveedor)` | más simple, pero un proceso deja de ser algo que puedas recorrer o conectar con otra cosa por sí mismo |

### Función usada: `MERGE`

- **Para qué sirve:** encuentra un nodo o relación que ya existe con esa identidad, o lo crea si no existe — nunca lo duplica.
- **Por qué aparece:** el runtime de Colab se reinicia solo, y el receso pasa a mitad de sesión. Sin `MERGE`, volver a ejecutar la carga crearía un segundo `Proceso 2024-001` idéntico al primero.
- **Intuición en palabras:** es como decir “busca esta persona por su cédula; si no está, regístrala — pero nunca la registres dos veces”.
- **Error frecuente:** usar `CREATE` en su lugar y terminar con varios nodos duplicados del mismo proceso.

### Cypher mínimo

| Construcción | Para qué sirve | Qué devuelve/cambia | Error frecuente |
|---|---|---|---|
| `MERGE` | encuentra o crea un patrón | nodos/relaciones persistidos | creer que siempre crea otro nodo |
| `MATCH` | busca patrones | filas con coincidencias | leerlo como un `SELECT *` sin relaciones |
| `WHERE` | filtra | menos coincidencias | filtrar antes de entender el patrón |
| `WITH` | encadena etapas | variables para la etapa siguiente | olvidar qué variables siguen vivas |
| `RETURN` | define la salida | columnas del resultado | confundir salida con persistencia |
| `ORDER BY` / `LIMIT` | ordena y acota | resultado priorizado | asumir orden si no se pidió |

**PARA LLEVAR.** La flecha es parte de la consulta: no es decoración visual.

In [ ]:
#@title Autoevaluación 4 — Cypher { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA0LCAidGVtYSI6ICJDeXBoZXIiLCAicHJlZ3VudGEiOiAiwr9Qb3IgcXXDqSB1c2FyZW1vcyBNRVJHRSB5IHJlc3RyaWNjaW9uZXMgw7puaWNhcz8iLCAib3BjaW9uZXMiOiBbIlBhcmEgcG9kZXIgcmVwZXRpciBsYSBjYXJnYSBzaW4gZmFicmljYXIgZHVwbGljYWRvcyBkZWwgbWlzbW8gaWRlbnRpZmljYWRvci4iLCAiUG9ycXVlIENSRUFURSBubyBwdWVkZSBjcmVhciByZWxhY2lvbmVzLiIsICJQb3JxdWUgTUVSR0UgZGVjaWRlIGVsIG1vZGVsbyBwb3Igbm9zb3Ryb3MuIl0sICJjb3JyZWN0YSI6IDAsICJyZXRybyI6IFsiQ29ycmVjdG8uIExhIGlkZW50aWRhZCBleHBsw61jaXRhIGhhY2UgbGEgY2FyZ2EgcmVwZXRpYmxlLiIsICJDUkVBVEUgc8OtIHB1ZWRlIGNyZWFyIHJlbGFjaW9uZXMuIiwgIkVsIG1vZGVsbyBzaWd1ZSBzaWVuZG8gdW5hIGRlY2lzacOzbiBodW1hbmEuIl0sICJjb250ZXh0byI6ICJBcGxpY2EgbG8gcXVlIGFjYWJhcyBkZSBvYnNlcnZhciBlbiBlbCBjYXNvIGRlIExhdXJhLiIsICJ0b3RhbCI6IG51bGx9")

---
## 4. Contrato de resultado: primero pandas

Antes de usar Neo4j calculamos qué proveedores de la entidad de trabajo también aparecen en otras entidades del extracto. Luego exigiremos a Neo4j la misma respuesta.

In [ ]:
prov_ancla = (
    hist_ancla.groupby(["nit_proveedor", "proveedor"], dropna=False)["id_proceso"]
    .nunique().rename("procesos_con_entidad").reset_index()
)
prov_global = (
    hist.groupby(["nit_proveedor", "proveedor"], dropna=False)["nit_entidad"]
    .nunique().rename("entidades_conectadas").reset_index()
)
esperado_pd = (
    prov_ancla.merge(prov_global, on=["nit_proveedor", "proveedor"], how="left")
    .sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True])
    .head(10).reset_index(drop=True)
)

MEDIANA_H2R = float(manifest.get("mediana_maximo_conectadas_candidatas", 0))
if esperado_pd.empty:
    desenlace_h2r_pd = "no evaluable con esta ancla"
elif esperado_pd["entidades_conectadas"].max() > MEDIANA_H2R:
    desenlace_h2r_pd = "conexión más fuerte que la mediana de las candidatas de S5"
else:
    desenlace_h2r_pd = "conexión igual o menor que la mediana de las candidatas de S5"

print("Mediana de referencia (candidatas S5):", MEDIANA_H2R)
print("Desenlace H2-R (pandas):", desenlace_h2r_pd)
esperado_pd

### Interpretación del contrato pandas y del desenlace H2-R

**Cómo se lee.** `procesos_con_entidad` cuenta procesos adjudicados de la entidad de trabajo; `entidades_conectadas` cuenta entidades distintas asociadas al mismo NIT de proveedor. El desenlace H2-R compara el máximo contra la mediana de esa misma métrica entre las 32 entidades candidatas de S5 que tienen historial.

**Qué nos dice.** Ya sabemos qué salida debería reproducir el grafo, y ya tenemos un desenlace declarado para H2-R.

**Qué NO permite concluir todavía.** Repetición o conectividad no equivale a favorecimiento, colusión ni irregularidad.

**Error frecuente.** Llamar “sospechoso” al proveedor que queda primero, o llamar “aceptada”/“rechazada” al desenlace de H2-R.

### RECUPERACIÓN S06 — si Colab reinició antes de Aura

Ejecuta la siguiente celda siempre que vuelvas del receso. Si el estado sigue vivo, solo lo confirma. Si se perdió, reconstruye todo desde cero automáticamente — sin pedirte ningún archivo.

In [ ]:
#@title Recuperar estado S6 { display-mode: "form" }
# RECUPERACIÓN S06
if "pregunta_codificada" not in globals() or "tutorial" not in globals():
    exec('\nimport base64, json, html as html_lib\nfrom IPython.display import display, HTML\n\ndef pregunta_codificada(token):\n    p = json.loads(base64.b64decode(token).decode("utf-8"))\n    uid = f"s06-p{p[\'numero\']}"\n    opts = "".join(\n        f\'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>\'\n        for i, op in enumerate(p["opciones"])\n    )\n    # Escapar el atributo COMPLETO; la retroalimentación se inserta como texto.\n    handler = (\n        "const box=this.closest(\'[data-pregunta]\');"\n        "const e=box.querySelector(\'input:checked\');"\n        "const s=box.querySelector(\'[aria-live]\');"\n        "if(!e){s.textContent=\'Selecciona una opción.\';return;}"\n        f"const i=Number(e.value),r={json.dumps(p[\'retro\'], ensure_ascii=False)};"\n        f"const ok=i==={p[\'correcta\']};"\n        "s.textContent=(ok?\'Correcto. \':\'Incorrecto. \')+r[i];"\n        "s.style.background=ok?\'#dcfce7\':\'#fee2e2\';"\n        "s.style.color=ok?\'#14532d\':\'#7f1d1d\';"\n        "s.style.padding=\'12px\';"\n    )\n    handler = html_lib.escape(handler, quote=True)\n    contador = str(p[\'numero\']) + (f" de {p[\'total\']}" if p.get(\'total\') else \'\')\n    box = (\n        f\'<div data-pregunta="{uid}" style="border:2px solid #1e40af;background:#eff6ff;color:#172554;border-radius:12px;padding:15px;margin:14px 0">\'\n        f\'<strong>Pregunta {contador} — {html_lib.escape(p["tema"])}</strong>\'\n        f\'<p style="background:#fef3c7;color:#713f12;padding:10px">{html_lib.escape(p.get("contexto", "Aplica lo que acabas de observar en el caso de Laura."))}</p>\'\n        f\'<p>{html_lib.escape(p["pregunta"])}</p>{opts}\'\n        f\'<button onclick="{handler}" \'\n        f\'style="background:#1e40af;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar respuesta</button>\'\n        f\'<div id="r-{uid}" aria-live="polite"></div></div>\'\n    )\n    display(HTML(box))\n\ndef tutorial(url, alto=720):\n    box = f\'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>\'\n    box += f\'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>\'\n    display(HTML(box))\n\nprint("Soporte S6 listo.")\n')

estado_necesario = ["datos", "manifest", "ancla_trabajo", "hist", "hist_ancla", "esperado_pd", "nit_deseado", "desenlace_h2r_pd"]
if not all(nombre in globals() for nombre in estado_necesario):
    import json, urllib.request
    import pandas as pd

    DATA_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'
    MANIFEST_URL = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json'
    datos = pd.read_csv(DATA_URL, low_memory=False)
    with urllib.request.urlopen(MANIFEST_URL) as r:
        manifest = json.loads(r.read().decode("utf-8"))

    ancla_trabajo = dict(manifest["ancla_pedagogica"])
    nit_deseado = str(ancla_trabajo.get("nit_entidad", "")).strip()
    hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
    hist_ancla = hist[hist["nit_entidad"].astype(str).str.strip().eq(nit_deseado)]

    prov_ancla = (
        hist_ancla.groupby(["nit_proveedor", "proveedor"], dropna=False)["id_proceso"]
        .nunique().rename("procesos_con_entidad").reset_index()
    )
    prov_global = (
        hist.groupby(["nit_proveedor", "proveedor"], dropna=False)["nit_entidad"]
        .nunique().rename("entidades_conectadas").reset_index()
    )
    esperado_pd = (
        prov_ancla.merge(prov_global, on=["nit_proveedor", "proveedor"], how="left")
        .sort_values(["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"], ascending=[False, False, True])
        .head(10).reset_index(drop=True)
    )

    MEDIANA_H2R = float(manifest.get("mediana_maximo_conectadas_candidatas", 0))
    if esperado_pd.empty:
        desenlace_h2r_pd = "no evaluable con esta ancla"
    elif esperado_pd["entidades_conectadas"].max() > MEDIANA_H2R:
        desenlace_h2r_pd = "conexión más fuerte que la mediana de las candidatas de S5"
    else:
        desenlace_h2r_pd = "conexión igual o menor que la mediana de las candidatas de S5"
    print("Estado S6 reconstruido desde archivos versionados.")
else:
    print("Estado S6 sigue en memoria; no fue necesario reconstruirlo.")

print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Filas contrato pandas:", len(esperado_pd))
print("Desenlace H2-R (pandas):", desenlace_h2r_pd)

---
## receso

## 5. Tutorial visual — AuraDB

**HAZ ESTO AHORA.** Vuelve cuando `RETURN 1 AS conexion` funcione en Query y tengas URI, usuario y contraseña.

El HTML es **instrumental**: muestra el camino de interfaz. Las pantallas dibujadas están rotuladas como representaciones; no se presentan como capturas autenticadas.

In [ ]:
#@title Abrir tutorial Neo4j Aura { display-mode: "form" }
tutorial('https://jazaineam1.github.io/BigData2026/assets/tutoriales/neo4j-aura-s06-paso-a-paso.html')

In [ ]:
!pip install -q "neo4j>=6,<7"
from getpass import getpass
from neo4j import GraphDatabase

URI = input("Connection URI: ").strip()
USER = input("User name: ").strip()
PASSWORD = getpass("Password (no se muestra): ")
if not URI or not USER or not PASSWORD:
    raise ValueError("URI, usuario y contraseña son obligatorios.")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("Conexión Neo4j verificada.")

---
## Antes de cargar los datos reales: un grafo de juguete

**Calentamiento; no es evidencia de tu hito.** Antes de cargar 2.109 filas de Compras Claras, practica los mismos movimientos de Cypher con un grafo mínimo y conocido: una película, tres actores, y una red de amigos. Si algo falla aquí, es mucho más fácil de diagnosticar que si falla con el dataset real.

### 1. Crear un nodo — `MERGE`

Ya conoces `MERGE` de la mini-ficha anterior. Ahora lo usas por primera vez contra Aura, con un ejemplo mínimo.

In [ ]:
driver.execute_query("MERGE (:Movie {title: $title})", title="The Matrix")
driver.execute_query("MERGE (:Person {name: $name})", name="Keanu Reeves")
driver.execute_query("MERGE (:Person {name: $name})", name="Carrie-Anne Moss")
driver.execute_query("MERGE (:Person {name: $name})", name="Laurence Fishburne")
print("Nodos de juguete creados: 1 Movie, 3 Person.")

### 2. Crear una relación — patrón `MATCH` + `MATCH` + `MERGE`

Para conectar dos nodos que ya existen, primero los *encuentras* con `MATCH` (uno por cada lado) y luego creas el puente entre ellos con `MERGE`.

In [ ]:
for actor in ["Keanu Reeves", "Carrie-Anne Moss", "Laurence Fishburne"]:
    driver.execute_query('''
        MATCH (actor:Person {name:$name})
        MATCH (pelicula:Movie {title:$title})
        MERGE (actor)-[:ACTED_IN]->(pelicula)
    ''', name=actor, title="The Matrix")
print("Relaciones ACTED_IN creadas para los 3 actores.")

**Cómo se lee.** Cada `MATCH` encuentra un nodo que ya existía; `MERGE` no vuelve a crearlo, solo agrega la flecha entre los dos.

**Qué nos dice.** Es el mismo patrón de tres pasos (encontrar, encontrar, conectar) que usarás con Entidad → Proceso → Proveedor.

**Qué NO permite concluir todavía.** Que dos nodos estén conectados no dice nada sobre la calidad de esa conexión — apenas estamos practicando la mecánica.

**Error frecuente.** Usar `MERGE` también para los dos `MATCH` — eso arriesga crear un actor o película duplicados si el nombre no coincide exactamente.

**HAZ ESTO AHORA.** Ejecuta la siguiente celda: va a **dibujar el grafo aquí mismo, en Colab** — nodos y flechas de verdad, con los datos que tú acabas de crear. No necesitas salir a Aura para esto.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

resultado_grafo = driver.execute_query('''
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    RETURN p.name AS persona, m.title AS pelicula
''')

G = nx.DiGraph()
for r in resultado_grafo.records:
    G.add_edge(r["persona"], r["pelicula"])

plt.figure(figsize=(6, 4))
pos = nx.spring_layout(G, seed=7)
colores = ["#c9a227" if n == "The Matrix" else "#175c3c" for n in G.nodes()]
nx.draw(G, pos, with_labels=True, node_color=colores, font_color="white", font_size=9, font_weight="bold", node_size=2400, edgecolors="black")
nx.draw_networkx_edge_labels(G, pos, edge_labels={e: "ACTED_IN" for e in G.edges()}, font_size=8)
plt.title("Tu primer grafo dibujado: actores → película")
plt.axis("off")
plt.show()

**OJO.** Son apenas 4 nodos — el "poder" de un grafo no está en que se vea bonito con pocos datos, está en que la MISMA consulta, sin cambiar una palabra, funcionaría igual de bien con 3 millones de actores y películas. Eso es justo lo que vas a comprobar más adelante con 2.109 filas reales.

### 3. Actualizar una propiedad — `SET`

`SET` agrega o cambia una propiedad de un nodo que ya existe. No crea nada nuevo.

In [ ]:
driver.execute_query("MATCH (p:Person {name:$name}) SET p.age = $age", name="Keanu Reeves", age=41)
driver.execute_query("MATCH (p:Person {name:$name}) SET p.age = $age", name="Laurence Fishburne", age=52)
print("Edad asignada a dos actores.")

### 4. Consultar — `MATCH` + `RETURN`, con filtros y conteo

Cuatro consultas, cada una agregando algo nuevo.

In [ ]:
r1 = driver.execute_query("MATCH (p:Person) RETURN p.name AS name")
print("Todas las personas:", [r["name"] for r in r1.records])

r2 = driver.execute_query("MATCH (p:Person {age:$age}) RETURN p.name AS name", age=41)
print("Personas de 41 años:", [r["name"] for r in r2.records])

r3 = driver.execute_query('''
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    RETURN p.name AS actor, m.title AS pelicula
''')
print("Quién actuó en qué:", [r.data() for r in r3.records])

r4 = driver.execute_query("MATCH (p:Person) WHERE p.age > 40 RETURN count(p) AS mayores_40")
print("Personas mayores de 40:", r4.records[0]["mayores_40"])

**Cómo se lee.** Cada consulta agrega una pieza: filtrar por propiedad, recorrer una relación, contar con una condición.

**Qué nos dice.** Con las mismas piezas (`MATCH`, `WHERE`, recorrer una relación, `count()`) vas a construir la consulta real de Compras Claras más adelante.

**Qué NO permite concluir todavía.** Nada — este grafo es de juguete y no representa ningún caso real.

**Error frecuente.** Olvidar que `age` es un número, no un texto: `age:"41"` no encontraría nada.

### 5. Borrar — `DETACH DELETE`

Borrar un nodo que tiene relaciones falla con `DELETE` a secas — hay que borrar también sus relaciones en el mismo paso, con `DETACH DELETE`.

In [ ]:
driver.execute_query("MATCH (p:Person {name:$name}) DETACH DELETE p", name="Laurence Fishburne")
check = driver.execute_query("MATCH (p:Person {name:$name}) RETURN p.name AS name", name="Laurence Fishburne")
print("¿Sigue existiendo Laurence Fishburne?", len(check.records) > 0)

**Cómo se lee.** `DETACH DELETE` borra el nodo y, en el mismo paso, todas sus relaciones — aquí, su `ACTED_IN` hacia The Matrix.

**Qué nos dice.** La consulta de verificación debe devolver una lista vacía: el nodo ya no está.

**Qué NO permite concluir todavía.** No aplica — es un borrado de práctica, no una decisión sobre datos reales.

**Error frecuente.** Usar `DELETE p` sin `DETACH` cuando el nodo todavía tiene relaciones — Neo4j lo rechaza con un error explícito en vez de borrar a medias.

### 6. Carga masiva desde Python — el mismo patrón `UNWIND` que usarás con 2.109 filas

Hasta ahora creaste nodos uno por uno. Cuando los datos ya viven en una lista de Python, `UNWIND` los recorre todos en una sola consulta — exactamente lo que vas a hacer con el extracto real en un momento.

In [ ]:
amigos = [
    {"name": "Alice", "age": 42, "friends": ["Bob", "Peter", "Anna"]},
    {"name": "Bob", "age": 19},
    {"name": "Peter", "age": 50},
    {"name": "Anna", "age": 30},
]

driver.execute_query('''
    UNWIND $filas AS fila
    MERGE (p:Person {name: fila.name})
    SET p.age = fila.age
''', filas=amigos)

con_amigos = [a for a in amigos if a.get("friends")]
driver.execute_query('''
    UNWIND $filas AS fila
    MATCH (p:Person {name: fila.name})
    UNWIND fila.friends AS nombre_amigo
    MATCH (amigo:Person {name: nombre_amigo})
    MERGE (p)-[:KNOWS]->(amigo)
''', filas=con_amigos)

print("Red de amigos cargada: 4 personas, relaciones KNOWS desde Alice.")

**Cómo se lee.** El primer `UNWIND` crea las 4 personas de una vez; el segundo recorre, para cada persona, su lista de amigos y crea la relación `KNOWS`.

**Qué nos dice.** Es exactamente la misma mecánica que usarás para cargar 2.109 filas de Compras Claras en un momento — la única diferencia es el tamaño de la lista.

**Qué NO permite concluir todavía.** Nada — sigue siendo el grafo de juguete.

**Error frecuente.** Anidar `UNWIND` dentro de `UNWIND` sin distinguir bien las variables — aquí `fila` y `nombre_amigo` son cosas distintas, no las confundas.

### Extra: una consulta que una tabla no responde tan fácil — "amigos de amigos"

En SQL esto pide un `JOIN` de la tabla contra sí misma. En Cypher es una flecha más en el mismo patrón.

In [ ]:
r5 = driver.execute_query('''
    MATCH (yo:Person {name:$name})-[:KNOWS]->(amigo)-[:KNOWS]->(amigo_de_amigo)
    WHERE amigo_de_amigo <> yo
    RETURN DISTINCT amigo_de_amigo.name AS nombre
''', name="Alice")
print("Amigos de amigos de Alice (2 saltos):", [r["nombre"] for r in r5.records])

**Cómo se lee.** La consulta recorre dos flechas `KNOWS` seguidas: de Alice a su amigo, y de ese amigo a los suyos.

**Qué nos dice.** Como en este grafo nadie tiene un segundo salto todavía (Bob, Peter y Anna no tienen amigos propios cargados), la lista sale vacía — y eso también es una lectura válida: el patrón está bien escrito, simplemente el dato no lo sostiene.

**Qué NO permite concluir todavía.** Nada nuevo — sigue siendo el grafo de juguete.

**Error frecuente.** Pensar que una lista vacía significa que la consulta está mal. Antes de asumir un error, confirma si el patrón realmente tiene datos que lo satisfagan.

**HAZ ESTO AHORA.** Dibuja el grafo completo de amigos, aquí mismo en Colab:

In [ ]:
resultado_amigos = driver.execute_query('''
    MATCH (p:Person)-[:KNOWS]-(otra:Person)
    RETURN p.name AS persona, otra.name AS otra_persona
''')

G_amigos = nx.Graph()
for r in resultado_amigos.records:
    G_amigos.add_edge(r["persona"], r["otra_persona"])

plt.figure(figsize=(6, 4))
pos = nx.spring_layout(G_amigos, seed=3)
colores_amigos = ["#c9a227" if n == "Alice" else "#3b5bab" for n in G_amigos.nodes()]
nx.draw(G_amigos, pos, with_labels=True, node_color=colores_amigos, font_color="white", font_size=9, font_weight="bold", node_size=2200, edgecolors="black")
plt.title("Red de amigos completa (KNOWS)")
plt.axis("off")
plt.show()

**PARA LLEVAR.** Con solo 7 nodos ya viste tres formas distintas de "preguntar por relaciones": contar cuántas salen de alguien, seguir dos saltos seguidos, y dibujar el grafo completo. Esas mismas tres formas son las que vas a usar en Compras Claras, con miles de nodos en vez de 7.

### Limpieza antes del caso real

Antes de cargar Compras Claras, borra el grafo de juguete completo para que no se mezcle con Entidad/Proceso/Proveedor.

In [ ]:
driver.execute_query("MATCH (n) WHERE n:Movie OR n:Person DETACH DELETE n")
check = driver.execute_query("MATCH (n) WHERE n:Movie OR n:Person RETURN count(n) AS restantes")
print("Nodos de juguete restantes (debe ser 0):", check.records[0]["restantes"])

---
## 6. Identidad y carga idempotente

Primero creamos restricciones. Después `UNWIND` recibe una lista de filas desde Python y `MERGE` reutiliza nodos ya existentes.

In [ ]:
constraints = [
    "CREATE CONSTRAINT entidad_nit IF NOT EXISTS FOR (e:Entidad) REQUIRE e.nit IS UNIQUE",
    "CREATE CONSTRAINT proceso_id IF NOT EXISTS FOR (p:Proceso) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT proveedor_nit IF NOT EXISTS FOR (v:Proveedor) REQUIRE v.nit IS UNIQUE",
]
for q in constraints:
    driver.execute_query(q)
print("Restricciones listas.")

### Función usada: `UNWIND`

- **Para qué sirve:** convierte una lista (de filas, de diccionarios) en filas individuales que Cypher procesa una por una dentro de la misma consulta.
- **Por qué aparece:** vas a cargar miles de filas de una sola vez desde Python; sin `UNWIND` tendrías que enviar una consulta por fila.
- **Intuición en palabras:** es como decir “toma esta lista de invitados y preséntamelos uno por uno”, para hacer lo mismo con cada uno.
- **Ejemplo manual:** con `filas = [{"id":"P1"}, {"id":"P2"}, {"id":"P3"}]`, `UNWIND $filas AS fila` hace que la consulta se ejecute tres veces: una con `fila.id = "P1"`, otra con `"P2"`, otra con `"P3"`.
- **Error frecuente:** usar `filas` (la lista completa) en vez de `fila` (el elemento actual) dentro del patrón.

In [ ]:
cols = [
    "entidad", "nit_entidad", "departamento_entidad", "id_proceso", "referencia",
    "nombre_proceso", "descripcion", "precio_base", "modalidad", "proveedor",
    "nit_proveedor", "departamento_proveedor", "noticias_entidad", "nivel_menciones",
    "url_secop", "es_proceso_candidato_s05", "es_entidad_candidata_s05",
]
rows = datos[cols].where(pd.notna(datos[cols]), None).to_dict("records")

query_base = '''
UNWIND $filas AS fila
MERGE (e:Entidad {nit: toString(fila.nit_entidad)})
SET e.nombre = fila.entidad,
    e.departamento = fila.departamento_entidad,
    e.es_candidata_s05 = fila.es_entidad_candidata_s05,
    e.noticias_entidad = fila.noticias_entidad,
    e.nivel_menciones = fila.nivel_menciones
MERGE (p:Proceso {id: fila.id_proceso})
SET p.referencia = fila.referencia,
    p.nombre = fila.nombre_proceso,
    p.descripcion = fila.descripcion,
    p.valor = fila.precio_base,
    p.modalidad = fila.modalidad,
    p.url = fila.url_secop,
    p.es_candidato_s05 = fila.es_proceso_candidato_s05
MERGE (e)-[:PUBLICA]->(p)
'''
driver.execute_query(query_base, filas=rows)

rows_proveedor = [r for r in rows if r.get("nit_proveedor")]
query_proveedor = '''
UNWIND $filas AS fila
MATCH (p:Proceso {id: fila.id_proceso})
MERGE (v:Proveedor {nit: toString(fila.nit_proveedor)})
SET v.nombre = fila.proveedor, v.departamento = fila.departamento_proveedor
MERGE (p)-[:ADJUDICADO_A]->(v)
'''
driver.execute_query(query_proveedor, filas=rows_proveedor)
print("Carga lista:", len(rows), "filas;", len(rows_proveedor), "adjudicaciones.")

### Antes de la consulta completa: qué agrega cada salto

Antes de la consulta final, mira qué cambia cuando agregas una flecha más al patrón.

In [ ]:
r0 = driver.execute_query("MATCH (e:Entidad) RETURN e.nombre AS entidad LIMIT 5")
print("0 relaciones -- solo nodos Entidad:")
print(pd.DataFrame([r.data() for r in r0.records]))

r1 = driver.execute_query("MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso) RETURN e.nombre AS entidad, p.id AS proceso LIMIT 5")
print("\n1 relacion (PUBLICA) -- que publico cada entidad:")
print(pd.DataFrame([r.data() for r in r1.records]))

In [ ]:
r2 = driver.execute_query('''
MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
RETURN e.nombre AS entidad, p.id AS proceso, v.nombre AS proveedor
LIMIT 5
''')
print("2 relaciones (PUBLICA + ADJUDICADO_A) -- a quien se adjudico:")
pd.DataFrame([r.data() for r in r2.records])

| Saltos | Qué responde | Ejemplo de pregunta |
|---|---|---|
| 0 | qué entidades existen | ¿qué entidades cargamos? |
| 1 (`PUBLICA`) | qué publicó cada entidad | ¿qué procesos abrió esta entidad? |
| 2 (`PUBLICA`+`ADJUDICADO_A`) | a quién se le adjudicó lo publicado | ¿a qué proveedor llegó este proceso? |

La consulta que sigue agrega un tercer salto: desde ese proveedor, vuelve a **todas** las entidades conectadas.

---
## 7. La consulta que justifica Neo4j

Ahora recorremos el patrón Entidad → Proceso → Proveedor y, desde ese proveedor, contamos otras entidades conectadas.

In [ ]:
query_contexto = '''
MATCH (e:Entidad {nit:$nit})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH v, count(DISTINCT p) AS procesos_con_entidad
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
RETURN v.nit AS nit_proveedor,
       v.nombre AS proveedor,
       procesos_con_entidad,
       count(DISTINCT otra) AS entidades_conectadas
ORDER BY entidades_conectadas DESC, procesos_con_entidad DESC, nit_proveedor ASC
LIMIT 10
'''
neo = driver.execute_query(query_contexto, nit=nit_deseado)
neo_df = pd.DataFrame([r.data() for r in neo.records])
neo_df

In [ ]:
def normalizar_nit(serie):
    # "123.0" y "123" deben tratarse como el mismo NIT (una conversión de
    # tipo intermedia puede volver flotante un entero antes de cargarlo).
    return serie.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

cols_cmp = ["nit_proveedor", "procesos_con_entidad", "entidades_conectadas"]
pd_cmp = esperado_pd[cols_cmp].copy()
neo_cmp = neo_df[cols_cmp].copy()
pd_cmp["nit_proveedor"] = normalizar_nit(pd_cmp["nit_proveedor"])
neo_cmp["nit_proveedor"] = normalizar_nit(neo_cmp["nit_proveedor"])
coinciden = pd_cmp.reset_index(drop=True).equals(neo_cmp.reset_index(drop=True))
print("pandas == Neo4j:", coinciden)

if not coinciden:
    print("\nFilas esperadas (pandas):")
    print(pd_cmp.reset_index(drop=True))
    print("\nFilas obtenidas (Neo4j):")
    print(neo_cmp.reset_index(drop=True))
    if len(neo_cmp) < len(pd_cmp):
        print("\nNeo4j devolvió MENOS filas que pandas — la carga de 2.109 filas probablemente no terminó.")
        print("Vuelve a ejecutar la celda 'Carga lista: ...' y espera a que termine antes de continuar.")

assert coinciden, "La respuesta Neo4j no coincide con el contrato pandas."

### Interpretación pandas ↔ Neo4j

**Cómo se lee.** Comparamos NIT y las dos métricas en el mismo orden.

**Qué nos dice.** El grafo reproduce el patrón calculado previamente.

**Qué NO permite concluir todavía.** Es una prueba de corrección, no un benchmark de velocidad ni evidencia de irregularidad.

**Error frecuente.** Confundir “la consulta coincide” con “Neo4j es más rápido”.

In [ ]:
#@title Autoevaluación 5 — Interpretación { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA1LCAidGVtYSI6ICJJbnRlcnByZXRhY2nDs24iLCAicHJlZ3VudGEiOiAiVW4gcHJvdmVlZG9yIGFwYXJlY2UgY29uZWN0YWRvIGNvbiBjdWF0cm8gZW50aWRhZGVzLiDCv1F1w6kgcHVlZGUgYWZpcm1hciBMYXVyYT8iLCAib3BjaW9uZXMiOiBbIlF1ZSBleGlzdGUgdW5hIHJlbGFjacOzbiBjb250cmFjdHVhbCBvYnNlcnZhZGEgY29uIHByb2Nlc29zIGRlIGN1YXRybyBlbnRpZGFkZXMgZGVudHJvIGRlbCBleHRyYWN0by4iLCAiUXVlIGxhcyBjdWF0cm8gZW50aWRhZGVzIGNvb3JkaW5hcm9uIHN1cyBhZGp1ZGljYWNpb25lcy4iLCAiUXVlIGVsIHByb3ZlZWRvciBpbmN1cnJpw7MgZW4gdW5hIGlycmVndWxhcmlkYWQuIl0sICJjb3JyZWN0YSI6IDAsICJyZXRybyI6IFsiQ29ycmVjdG8uIEVsIGdyYWZvIGRlc2NyaWJlIGVzdHJ1Y3R1cmEgcmVnaXN0cmFkYS4iLCAiTGEgY29uZWN0aXZpZGFkIHBvciBzw60gc29sYSBubyBwcnVlYmEgY29vcmRpbmFjacOzbi4iLCAiTGEgY29uZWN0aXZpZGFkIHBvciBzw60gc29sYSBubyBwcnVlYmEgaXJyZWd1bGFyaWRhZC4iXSwgImNvbnRleHRvIjogIkFwbGljYSBsbyBxdWUgYWNhYmFzIGRlIG9ic2VydmFyIGVuIGVsIGNhc28gZGUgTGF1cmEuIiwgInRvdGFsIjogbnVsbH0=")

---
## Lo que un grafo puede hacer y una tabla no: el camino más corto

Hasta ahora contaste conexiones. Ahora vas a preguntar algo distinto: **¿cuál es el camino más corto entre tu entidad y otra, sin importar cuántos proveedores intermedios haga falta recorrer?** En SQL esto exige escribir un `JOIN` distinto por cada número de saltos que quieras probar, sin saber de antemano cuántos hacen falta. En Cypher es una sola palabra: `shortestPath`.

In [ ]:
if neo_df.empty:
    print("No hay proveedores conectados para calcular un camino.")
else:
    top_nit_proveedor = str(neo_df.iloc[0]["nit_proveedor"])
    otra_entidad = driver.execute_query('''
        MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$nit_proveedor})
        WHERE otra.nit <> $nit_propio
        RETURN otra.nit AS nit, otra.nombre AS nombre
        LIMIT 1
    ''', nit_proveedor=top_nit_proveedor, nit_propio=nit_deseado)

    if not otra_entidad.records:
        print("No se encontró otra entidad conectada para comparar caminos.")
    else:
        nit_destino_camino = str(otra_entidad.records[0]["nit"])
        nombre_destino_camino = otra_entidad.records[0]["nombre"]
        camino = driver.execute_query('''
            MATCH (origen:Entidad {nit:$nit_origen}), (destino:Entidad {nit:$nit_destino})
            MATCH ruta = shortestPath((origen)-[*..6]-(destino))
            RETURN [n IN nodes(ruta) | coalesce(n.nombre, n.id)] AS pasos, length(ruta) AS saltos
        ''', nit_origen=nit_deseado, nit_destino=nit_destino_camino)
        if camino.records:
            r = camino.records[0]
            print(f"Camino más corto hasta \"{nombre_destino_camino}\": {r['saltos']} saltos")
            print(" -> ".join(str(p) for p in r["pasos"]))
        else:
            print("No se encontró un camino en 6 saltos o menos.")

**Cómo se lee.** `shortestPath` explora el grafo por ti y devuelve la ruta más corta que conecta los dos nodos, sin que tengas que decidir de antemano cuántos saltos probar.

**Qué nos dice.** Existe al menos una cadena de relaciones registradas entre tu entidad y la otra — a través de procesos y proveedores intermedios.

**Qué NO permite concluir todavía.** Un camino corto no es lo mismo que una relación sospechosa. Conecta datos registrados; no mide intención ni coordinación.

**Error frecuente.** Confundir "camino más corto" con "relación más fuerte" — `shortestPath` no pesa las relaciones, solo cuenta saltos.

**HAZ ESTO AHORA.** Pega esto en la pestaña Query de tu instancia Aura para ver el camino dibujado, nodo por nodo:

In [ ]:
print(f'''
MATCH origen = (e:Entidad {{nit:"{nit_deseado}"}})
MATCH ruta = shortestPath((e)-[*..6]-(destino:Entidad))
WHERE destino.nit <> "{nit_deseado}"
RETURN ruta
LIMIT 1
''')

---
## Demostración guiada — el vecindario más rico del extracto

Ya calculaste el vecindario de esta misma entidad en pandas y en Neo4j. Ahora lo vas a **ver dibujado**, con nodos y flechas de verdad, en Aura.

In [ ]:
nit_ancla_demo = str(manifest["ancla_pedagogica"]["nit_entidad"]).strip()

query_demo_top = '''
MATCH (e:Entidad {nit:$nit})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH v, count(DISTINCT p) AS procesos_con_entidad
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
WITH v, procesos_con_entidad, count(DISTINCT otra) AS entidades_conectadas
WHERE entidades_conectadas > 1
RETURN v.nit AS nit_proveedor, v.nombre AS proveedor, procesos_con_entidad, entidades_conectadas
ORDER BY entidades_conectadas DESC, procesos_con_entidad DESC, nit_proveedor ASC
LIMIT 5
'''
neo_demo_df = pd.DataFrame([r.data() for r in driver.execute_query(query_demo_top, nit=nit_ancla_demo).records])
neo_demo_df

In [ ]:
top_demo = neo_demo_df.iloc[0]
nit_proveedor_demo = str(top_demo["nit_proveedor"])

query_visual_demo = f'''
MATCH camino_ancla = (e:Entidad {{nit:"{nit_ancla_demo}"}})-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {{nit:"{nit_proveedor_demo}"}})
WITH v, collect(camino_ancla)[0] AS camino_ancla
MATCH camino_otras = (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
WHERE otra.nit <> "{nit_ancla_demo}"
WITH camino_ancla, otra, collect(camino_otras)[0] AS camino_otras
RETURN camino_ancla, camino_otras
LIMIT 8
'''
print(query_visual_demo)

**HAZ ESTO AHORA.** La siguiente celda dibuja este vecindario **aquí mismo en Colab**, con el mismo proveedor y hasta 8 entidades conectadas.

In [ ]:
otras_entidades_demo = driver.execute_query('''
    MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$nit_proveedor})
    WHERE otra.nit <> $nit_ancla
    RETURN DISTINCT otra.nombre AS nombre
    LIMIT 8
''', nit_proveedor=nit_proveedor_demo, nit_ancla=nit_ancla_demo)

G_demo = nx.Graph()
nombre_ancla_demo = str(manifest["ancla_pedagogica"]["entidad"])
nombre_proveedor_demo = str(top_demo["proveedor"])
G_demo.add_edge(nombre_ancla_demo, nombre_proveedor_demo)
for r in otras_entidades_demo.records:
    G_demo.add_edge(r["nombre"], nombre_proveedor_demo)

plt.figure(figsize=(9, 7))
pos = nx.spring_layout(G_demo, seed=11, k=1.3)
colores_demo = []
for n in G_demo.nodes():
    if n == nombre_proveedor_demo:
        colores_demo.append("#c9a227")
    elif n == nombre_ancla_demo:
        colores_demo.append("#175c3c")
    else:
        colores_demo.append("#3b5bab")
nx.draw(G_demo, pos, with_labels=True, node_color=colores_demo, font_color="white", font_size=8, font_weight="bold", node_size=2600, edgecolors="black")
plt.title(f"Vecindario real de Compras Claras: {nombre_proveedor_demo}")
plt.axis("off")
plt.show()

**Cómo se lee.** El punto dorado es el proveedor; el verde es tu entidad ancla; los azules son las demás entidades conectadas al mismo proveedor.

**Opcional — verlo interactivo en Aura.** Si quieres poder arrastrar los nodos y hacer zoom, copia la consulta que imprimió la celda anterior, ve a tu instancia AuraDB → pestaña **Query** (no Colab), pégala y ejecútala ahí. Debería verse parecido a esto:



**OJO.** El grafo muestra hasta 8 de las entidades conectadas con este proveedor — se acota para que se pueda leer, no porque las demás no existan.

### Antes de seguir, dos preguntas para el salón

1. En el grafo que acabas de ver, ¿cuál es el nodo puente entre las distintas entidades?
2. ¿Qué representa cada camino que dibujó Aura, y qué NO demuestra por sí solo?

**PARA LLEVAR.** Más conexiones no es lo mismo que una conexión anómala.

---
## 8. CRUD seguro y tu propio vecindario

El CRUD usa `S06-DEMO`; no modificamos un proceso real. Después eliges uno de los **5 proveedores con más entidades conectadas** — esa elección sí es tuya, aunque la entidad de trabajo sea compartida.

In [ ]:
driver.execute_query('''
MERGE (e:Entidad {nit:'S06-E'}) SET e.nombre='Entidad demo'
MERGE (p:Proceso {id:'S06-DEMO'}) SET p.nombre='Proceso demo'
MERGE (v:Proveedor {nit:'S06-V'}) SET v.nombre='Proveedor demo'
MERGE (e)-[:PUBLICA]->(p)
MERGE (p)-[:ADJUDICADO_A]->(v)
''')
r = driver.execute_query("MATCH (p:Proceso {id:'S06-DEMO'}) SET p.estado_revision='revisado' RETURN p.estado_revision AS estado")
assert r.records[0]["estado"] == "revisado"
driver.execute_query("MATCH (n) WHERE n.nit IN ['S06-E','S06-V'] OR n.id='S06-DEMO' DETACH DELETE n")
print("CRUD demo completado y limpiado.")

In [ ]:
if neo_df.empty:
    raise ValueError("No hay proveedores para elegir.")
opciones = neo_df.head(5)
for i, row in opciones.iterrows():
    print(f"{i+1:>2}. {row['proveedor']} | entidades={row['entidades_conectadas']}")
sel = int(input("Número de proveedor: ").strip())
if not 1 <= sel <= len(opciones):
    raise ValueError("Número fuera de rango")
proveedor_elegido = opciones.iloc[sel-1]

vec = driver.execute_query('''
MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$nit})
RETURN e.nombre AS entidad, p.id AS proceso, p.nombre AS nombre_proceso, p.valor AS valor
ORDER BY entidad, valor DESC
''', nit=str(proveedor_elegido["nit_proveedor"]))
vecindario_df = pd.DataFrame([r.data() for r in vec.records])

print("Tamaño observable de tu vecindario:")
print("  Procesos con la entidad de trabajo:", int(proveedor_elegido["procesos_con_entidad"]))
print("  Entidades conectadas:", int(proveedor_elegido["entidades_conectadas"]))
print("  Procesos visibles en el vecindario:", len(vecindario_df))
vecindario_df

### Interpretación de tu vecindario

**Cómo se lee.** Cada fila es un proceso conectado al proveedor que elegiste; una misma entidad puede aportar varios procesos.

**Qué nos dice.** Puedes observar qué entidades y procesos del extracto comparten ese actor contractual.

**Qué NO permite concluir todavía.** Compartir proveedor no demuestra coordinación, favorecimiento ni irregularidad.

**Error frecuente.** Convertir el número de conexiones en un “score de riesgo” sin modelo ni denominador.

### La evidencia no termina en el grafo

**OJO — esta versión no te pregunta el límite ni la alternativa.** Quedan fijos en el texto de abajo (revísalos, son reales para este caso), y por eso la rúbrica ya no los evalúa como criterio individual — lo que sigue midiendo tu propia ejecución es el proveedor que elegiste y el desenlace de H2-R.

In [ ]:
if neo_df.empty:
    desenlace_h2r_neo = "no evaluable con esta ancla"
elif neo_df["entidades_conectadas"].max() > MEDIANA_H2R:
    desenlace_h2r_neo = "conexión más fuerte que la mediana de las candidatas de S5"
else:
    desenlace_h2r_neo = "conexión igual o menor que la mediana de las candidatas de S5"

assert desenlace_h2r_neo == desenlace_h2r_pd, "El desenlace H2-R debería coincidir: ya vimos pandas == Neo4j."
print("Desenlace H2-R (Neo4j):", desenlace_h2r_neo)

In [ ]:
from pathlib import Path

limite_estudiante = "El extracto no incluye fechas de pago ni historial de cumplimiento contractual previo del proveedor."
alternativa_modelo = "Proceso como propiedad de la relación Entidad-Proveedor"
razon_alternativa = "Porque necesitamos que Proceso sea un nodo recorrible para S7, no solo un atributo"

export = vecindario_df.merge(
    datos[["id_proceso", "descripcion", "modalidad", "url_secop"]].drop_duplicates("id_proceso"),
    left_on="proceso", right_on="id_proceso", how="left"
)
export.to_json("s06_contexto_procesos.jsonl", orient="records", lines=True, force_ascii=False)

hito = f'''# Hito S06 (alterna) — Ficha relacional de revisión\n\n- Caso de trabajo: ancla pedagógica compartida (no una elección personal de S5)\n- Entidad: {ancla_trabajo.get("entidad", "")}\n- Proceso: {ancla_trabajo.get("id_proceso", "")}\n- Noticias / nivel: {ancla_trabajo.get("noticias_entidad", "")} / {ancla_trabajo.get("nivel_menciones", "")}\n- H1 (S5): 0/77 → refutada literalmente\n- H2-R (S6), desenlace pandas: {desenlace_h2r_pd}\n- H2-R (S6), desenlace Neo4j: {desenlace_h2r_neo}\n- pandas == Neo4j: {coinciden}\n- Proveedor elegido: {proveedor_elegido["proveedor"]}\n- Entidades conectadas: {int(proveedor_elegido["entidades_conectadas"])}\n- Procesos en el vecindario: {len(vecindario_df)}\n\n## Límite\n{limite_estudiante}\n\n## Decisión de modelado\nProceso se modeló como nodo porque participa en caminos y su texto será reutilizado en la siguiente sesión.\n\n### Alternativa descartada\n{alternativa_modelo}\n\nRazón: {razon_alternativa}\n'''
Path("hito_s06_ficha_relacional.md").write_text(hito, encoding="utf-8")
print(hito)

try:
    from google.colab import files
    files.download("hito_s06_ficha_relacional.md")
    files.download("s06_contexto_procesos.jsonl")
except Exception:
    print("Archivos generados en el runtime.")

## Rúbrica S06

| Criterio | Completo | Parcial | Sin evidencia | Peso |
|---|---|---|---|---:|
| Continuidad | identifica la entidad de trabajo y su historial | solo entidad | no ejecuta la celda | 15 |
| Modelo | resuelve el hueco `ADJUDICADO_A` y el ejercicio de vocabulario correctamente | solo uno de los dos | copia el patrón sin resolverlo | 20 |
| Ejecución | vecindario propio ejecutado | solo consulta común | no hay salida | 20 |
| Verificación | `pandas == Neo4j` comprobado | muestra ambos | solo uno | 15 |
| Evidencia propia | proveedor elegido + entidades + procesos + desenlace H2-R declarado | incompleta | genérica | 30 |

**OJO.** En esta versión el límite y la alternativa de modelado del hito vienen fijos en el cuaderno (no son texto del estudiante), así que no se califican como criterio individual — la evidencia propia se concentra en qué proveedor eligió cada quien y en su desenlace de H2-R.

Las autoevaluaciones son formativas. El hito es la evidencia revisable de la sesión.

---
## Hoja de trucos y puente

```text
UNWIND → convierte una lista en filas
MERGE  → encuentra o crea
MATCH  → busca patrón
WHERE  → filtra
WITH   → encadena
SET    → modifica una propiedad
RETURN → salida
DETACH DELETE → elimina nodo y relaciones
```



**Idea central.** Cassandra organizó datos para una pregunta repetitiva conocida. Neo4j hace de las relaciones una parte explícita de la pregunta.

### Lo que sigue

Laura ya puede ver el vecindario, pero ahora tiene muchos nombres y descripciones de procesos. La nueva pregunta será:

> **¿Cuáles de esos procesos son más relevantes para una búsqueda textual concreta?**

`s06_contexto_procesos.jsonl` será la entrada de Elasticsearch/BM25.

In [ ]:
try:
    driver.close()
    print("Conexión Neo4j cerrada.")
except Exception:
    pass